#1. Grundstruktur och Klasser
Här sätter vi upp våra godkända filterlistor samt definierar våra klassritningar för jobbannonser med arv.

In [ ]:
import csv
import os
import random
from datetime import datetime
import requests



accepted_languages = ["Python", "Java", "C++", "C#", "SQL"]
accepted_locations = ["Stockholm", "Göteborg", "Malmö"]

cleaned_data = []
my_dataset = []


class Jobb_annons:
    def __init__(self, language, location, date_created):
        self.language = language
        self.location = location
        self.date_created = date_created
        
#Koppling till yrkesrollen: Precis som Lovable gör, hjälper denna funktion att ta emot de värde som användaren ger och sedan gå igenom den godkända listan för att säkerställa att inga felaktiga värden når AI-modellen.
    def validate_language(self):
      return self.language in accepted_languages
        
    def validate_location(self):
      return self.location in accepted_locations

class Distans_jobb(Jobb_annons):
   def __init__(self, language, location, distans, date_created):
      super().__init__(language, location, date_created)
      self.distans = distans      





In [ ]:
os.makedirs(os.path.join("data", "raw"), exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("output", exist_ok=True)

url = "https://jobsearch.api.jobtechdev.se/search?q=Python&limit=10"

print("Anropar JobTech API...")

response = requests.get(url)

riktiga_annonser_lista = []

if response.status_code == 200:
  api_data =response.json()

  annonser = api_data.get("hits", [])

  tidsstampel = datetime.now().strftime("%Y-%m-%d %H:%M")

  for annons in annonser:
    titel = annons.get("headline", "Okänd titel")
    ort = annons.get("workplace_address", {}).get("municipality", "Okänd ort")

    ny_annons_objekt = Jobb_annons(language="Python", location=ort, date_created=tidsstampel)

    riktiga_annonser_lista.append(ny_annons_objekt)

  print(f"Lyckades hämta och skapa {len(riktiga_annonser_lista)} st jobbobjekt från API!\n")

else:
  print(f"Kunde inte hämta data från API. Felkod från servern: {response.status_code}")

  
csv_file_path = os.path.join("data", "raw", "raw_data.csv")

min_fil = open(csv_file_path, "w", encoding="utf-8")

min_fil.write("Language,Location,Date_Created\n")

for jobb_objekt in riktiga_annonser_lista:
  rad_text = f"{jobb_objekt.language},{jobb_objekt.location},{jobb_objekt.date_created}\n"
  min_fil.write(rad_text)

min_fil.close()

print(f"Framgång! Riktig data har sparats i: {csv_file_path}")


Jag använde mig av CSV som modul då de jobbannonser jag arbetar med gör det enklast att spara de i en tabell format för att göra det lätt läst och kunna dela vidare detta.